# Notebook 07 — Model Training with SMOTE

Train four base learners (Logistic Regression, Random Forest, XGBoost, LightGBM) on SMOTE-balanced training data using fixed reasonable parameters without hyperparameter tuning.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


In [ ]:
# Load SMOTE-balanced training data and untouched test data
X_train = joblib.load("data/processed/X_train_smote.pkl")
y_train = joblib.load("data/processed/y_train_smote.pkl")

test_df = pd.read_csv("data/processed/test_dataset.csv")
X_test = test_df.drop("Maintenance_Required", axis=1)
y_test = test_df["Maintenance_Required"]

print("SMOTE Training shape:", X_train.shape, y_train.shape)
print("Untouched Test shape:", X_test.shape, y_test.shape)


In [ ]:
# Define four base models with fixed reasonable default parameters (NO hyperparameter tuning)
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=-1,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
}


In [ ]:
# Train each base learner on SMOTE training data and evaluate on untouched test data
results = []

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

for name, model in models.items():
    print("=" * 60)
    print(f"Training {name} on SMOTE-balanced dataset...")
    print("=" * 60)
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC-AUC": roc
    })


In [ ]:
# Save all four trained base models individually
filename_map = {
    "Logistic Regression": "logistic_regression_smote.pkl",
    "Random Forest": "random_forest_smote.pkl",
    "XGBoost": "xgboost_smote.pkl",
    "LightGBM": "lightgbm_smote.pkl"
}

for name, model in models.items():
    fname = filename_map[name]
    joblib.dump(model, f"models/{fname}")
    print(f"Saved base model '{name}' to models/{fname}")


In [ ]:
# Save base model performance summary
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
results_df.to_csv("results/model_performance_smote.csv", index=False)

print("Base model SMOTE training results saved successfully:")
results_df
